In [ ]:
# ============================================================
# Cell 0: 环境自检（本章纯 CPU 即可，无需 GPU）
# ============================================================
# 本章只训一个很小的 GRU seq2seq（隐藏维 96、序列长 ≤ 12），全程 CPU 几分钟跑完，
# 所以这里不强制 GPU，只打印环境信息确认 PyTorch 可用。
import sys, platform
import torch

print("Python:", sys.version.split()[0])
print("平台:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available(), "（本章用不到，CPU 即可）")

In [ ]:
%%capture
# ============================================================
# Cell 1: 安装依赖
# ============================================================
# %%capture 必须是 cell 第一行，把 pip 安装日志藏起来
# torch:       张量运算 + nn.GRU——Colab 默认已装且够新，故意【不】加 -U：会话中途
#              升级 torch 容易让内核半新半旧后续 import 报错，本章只用基础算子无需新版。
# matplotlib:  画 attention 对齐矩阵的热力图、长度-准确率对比曲线
!pip install -q -U matplotlib

In [ ]:
# ============================================================
# Cell 2: RNN 一步前向，手算一遍隐藏状态怎么更新（对应第 1.2 节）
# ============================================================
# RNN 的核心就一条递推：h_t = tanh(W_hh h_{t-1} + W_xh x_t + b)。
# 这里用一个 2 维隐藏状态、3 维输入的玩具尺寸，手动走三步，看 h 怎么把历史「滚」进去。
import torch

torch.manual_seed(0)
d_in, d_h = 3, 2                       # 输入 3 维、隐藏 2 维
W_xh = torch.randn(d_h, d_in)          # [d_h, d_in] 输入到隐藏
W_hh = torch.randn(d_h, d_h)           # [d_h, d_h] 隐藏到隐藏（这就是「循环」的那条边）
b    = torch.zeros(d_h)

def rnn_step(x_t, h_prev):
    """单步 RNN：把上一步隐藏 h_prev 和当前输入 x_t 揉进新的隐藏 h_t。"""
    return torch.tanh(W_xh @ x_t + W_hh @ h_prev + b)

xs = [torch.randn(d_in) for _ in range(3)]   # 一条长度 3 的序列
h = torch.zeros(d_h)                          # 初始隐藏状态置 0
for t, x_t in enumerate(xs):
    h = rnn_step(x_t, h)                       # 同一组权重 W_xh/W_hh 反复用于每一步
    print(f"第 {t} 步后的隐藏状态 h = {h.numpy().round(3)}")
print("→ 注意：每一步都复用同一套权重，h 把『到目前为止读过的所有 token』压成一个定长向量")

In [ ]:
# ============================================================
# Cell 3: 构造 toy 任务「把数字串反过来」+ 词表（对应第 8 节）
# ============================================================
# 任务：输入一串随机数字（如 3 1 4 1 5），输出它的逆序（5 1 4 1 3）。
# 为什么选逆序？它逼着 decoder 第 t 步必须去「对齐」输入的第 (L-1-t) 个位置——
# 于是 attention 学到的对齐矩阵会是一条漂亮的反对角线，肉眼可验证。
import torch

PAD, SOS, EOS = 0, 1, 2                # 三个特殊 token
DIGIT0 = 3                             # 数字 0..9 映射到 token 3..12
VOCAB = 3 + 10                         # 词表大小 = 3 个特殊符 + 10 个数字 = 13

def make_batch(batch_size, min_len=4, max_len=12, seed=None):
    """随机生成一批「数字串 → 逆序」样本，长度在 [min_len, max_len] 间均匀取。
    返回:
      src      [B, Smax]  编码器输入（数字 token，右侧 PAD 补齐）
      src_len  [B]        每条源序列真实长度（不含 PAD）
      tgt_in   [B, Smax+1]  解码器输入（SOS + 逆序数字，teacher forcing 用）
      tgt_out  [B, Smax+1]  解码器目标（逆序数字 + EOS）
    """
    g = torch.Generator().manual_seed(seed) if seed is not None else None
    lens = torch.randint(min_len, max_len + 1, (batch_size,), generator=g)
    Smax = int(lens.max())
    src     = torch.full((batch_size, Smax), PAD, dtype=torch.long)
    tgt_in  = torch.full((batch_size, Smax + 1), PAD, dtype=torch.long)
    tgt_out = torch.full((batch_size, Smax + 1), PAD, dtype=torch.long)
    for i, L in enumerate(lens.tolist()):
        digits = torch.randint(0, 10, (L,), generator=g) + DIGIT0   # L 个数字 token
        rev = torch.flip(digits, dims=[0])                          # 逆序
        src[i, :L] = digits
        tgt_in[i, 0] = SOS                                          # decoder 第一步喂 SOS
        tgt_in[i, 1:L + 1] = rev
        tgt_out[i, :L] = rev
        tgt_out[i, L] = EOS                                         # 目标末尾是 EOS
    return src, lens, tgt_in, tgt_out

# 看一眼一个迷你 batch 长什么样
src, src_len, tgt_in, tgt_out = make_batch(3, min_len=4, max_len=6, seed=42)
print("src     (编码器输入, 数字token):\n", src)
print("src_len (真实长度):", src_len.tolist())
print("tgt_in  (解码器输入, 以 SOS=1 开头):\n", tgt_in)
print("tgt_out (解码器目标, 以 EOS=2 结尾):\n", tgt_out)

In [ ]:
# ============================================================
# Cell 4: 编码器 Encoder —— 一个 GRU 把源序列读成一串隐藏状态（对应第 2.2 节）
# ============================================================
import torch.nn as nn

class Encoder(nn.Module):
    """Embedding + 单层 GRU。返回每个位置的隐藏状态（attention 要用全部），
    以及最后一步的隐藏状态（无 attention 的 baseline 把它当唯一的 context）。

    关键细节：用 pack_padded_sequence 让 GRU 在每条序列【真实长度】处停下，不去
    处理右侧补的 PAD。否则 h_n 会变成「又跑了一串 PAD 之后」的隐藏态，还会随 batch
    里最长序列的长度而漂移——模型学歪、短序列上直接崩。这是 RNN 喂变长 batch 最常
    踩的坑，务必记住：别让 RNN 白跑 padding。"""
    def __init__(self, vocab, emb_dim, hid_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab, emb_dim, padding_idx=PAD)
        self.gru = nn.GRU(emb_dim, hid_dim, batch_first=True)

    def forward(self, src, src_len):
        emb = self.embed(src)                              # [B, S, emb]
        # 打包成「不含 PAD」的紧凑序列；enforce_sorted=False 允许长度不排序
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, src_len.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, h_n = self.gru(packed)
        # 解包回 [B, S, hid]，PAD 位置补 0（attention 会再屏蔽，不影响结果）
        outputs, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=src.size(1))
        return outputs, h_n                                # h_n: [1, B, hid] 真实末 token 隐藏

# 形状自检
enc = Encoder(VOCAB, emb_dim=32, hid_dim=96)
_out, _h = enc(src, src_len)
print("encoder outputs:", _out.shape, " (= [B, S, hid])")
print("encoder h_n:    ", _h.shape, " (= [1, B, hid]，每条序列真实末 token 的隐藏态)")

In [ ]:
# ============================================================
# Cell 5: Bahdanau（加性）注意力模块（对应第 4 节）
# ============================================================
import torch.nn.functional as F

class BahdanauAttention(nn.Module):
    """加性注意力：score(s, h_j) = v^T tanh(W s + U h_j)。
    输入 decoder 上一步隐藏 s [B, hid] 和 encoder 全部输出 H [B, S, hid]，
    输出 context [B, hid] 和注意力权重 alpha [B, S]（已对 PAD 位置屏蔽）。"""
    def __init__(self, hid_dim):
        super().__init__()
        self.W = nn.Linear(hid_dim, hid_dim, bias=False)   # 作用在 decoder 隐藏 s 上
        self.U = nn.Linear(hid_dim, hid_dim, bias=False)   # 作用在 encoder 每个 h_j 上
        self.v = nn.Linear(hid_dim, 1, bias=False)         # 把打分压成一个标量

    def forward(self, s, H, mask):
        # s: [B, hid] -> [B, 1, hid] 便于和 H 的 S 个位置广播相加
        # H: [B, S, hid]；mask: [B, S]，True 表示真实 token、False 表示 PAD
        energy = torch.tanh(self.W(s).unsqueeze(1) + self.U(H))   # [B, S, hid]
        scores = self.v(energy).squeeze(-1)                        # [B, S] 每个源位置一个分
        scores = scores.masked_fill(~mask, float("-inf"))         # PAD 位置打 -inf，softmax 后≈0
        alpha = F.softmax(scores, dim=-1)                          # [B, S] 注意力权重，和为 1
        context = torch.bmm(alpha.unsqueeze(1), H).squeeze(1)      # 加权求和 -> [B, hid]
        return context, alpha

In [ ]:
# ============================================================
# Cell 6: 解码器 Decoder —— 带 / 不带 attention 两种模式（对应第 2.3 / 4 节）
# ============================================================
class Decoder(nn.Module):
    """单步解码器（GRUCell）。use_attention 开关切换两种 context 来源：
      - use_attention=True : 每步用 Bahdanau attention 动态算 context（回看整句）
      - use_attention=False: context 恒为 encoder 最后一步隐藏态（固定瓶颈，baseline）
    GRUCell 的输入是 [上一步token embedding ; context] 拼接而成。"""
    def __init__(self, vocab, emb_dim, hid_dim, use_attention):
        super().__init__()
        self.use_attention = use_attention
        self.embed = nn.Embedding(vocab, emb_dim, padding_idx=PAD)
        self.attn = BahdanauAttention(hid_dim) if use_attention else None
        self.cell = nn.GRUCell(emb_dim + hid_dim, hid_dim)
        self.out = nn.Linear(hid_dim + hid_dim, vocab)   # 拼 [s ; context] 再投到词表

    def step(self, y_prev, s, H, mask, fixed_ctx):
        # y_prev: [B] 上一步 token；s: [B, hid] 上一步隐藏；H: [B, S, hid] 编码器输出
        emb = self.embed(y_prev)                          # [B, emb]
        if self.use_attention:
            context, alpha = self.attn(s, H, mask)        # [B, hid], [B, S]
        else:
            context, alpha = fixed_ctx, None              # 始终是同一个向量（瓶颈）
        s = self.cell(torch.cat([emb, context], dim=-1), s)   # [B, hid] 新隐藏
        logits = self.out(torch.cat([s, context], dim=-1))    # [B, vocab]
        return logits, s, alpha

In [ ]:
# ============================================================
# Cell 7: Seq2Seq 整体 + 训练/评估函数（对应第 8 节）
# ============================================================
class Seq2Seq(nn.Module):
    def __init__(self, vocab, emb_dim, hid_dim, use_attention):
        super().__init__()
        self.encoder = Encoder(vocab, emb_dim, hid_dim)
        self.decoder = Decoder(vocab, emb_dim, hid_dim, use_attention)

    def forward(self, src, src_len, tgt_in, return_attn=False):
        H, h_n = self.encoder(src, src_len)       # H: [B, S, hid], h_n: [1, B, hid]
        s = h_n.squeeze(0)                         # decoder 初始隐藏 = encoder 末隐藏 [B, hid]
        fixed_ctx = s                              # baseline 用的固定 context
        # mask: [B, S]，True=真实 token。arange < 长度 即为有效位置
        mask = torch.arange(src.size(1)).unsqueeze(0) < src_len.unsqueeze(1)
        logits_seq, attn_seq = [], []
        for t in range(tgt_in.size(1)):            # 逐步解码（teacher forcing 喂真值）
            logits, s, alpha = self.decoder.step(tgt_in[:, t], s, H, mask, fixed_ctx)
            logits_seq.append(logits)
            if return_attn and alpha is not None:
                attn_seq.append(alpha)
        logits = torch.stack(logits_seq, dim=1)    # [B, T, vocab]
        attn = torch.stack(attn_seq, dim=1) if (return_attn and attn_seq) else None
        return logits, attn                        # attn: [B, T, S] 或 None

def train(model, steps=2000, batch_size=64, lr=2e-3, log_every=400):
    """标准训练循环：前向 -> 交叉熵 loss（忽略 PAD）-> 反向 -> 优化器更新。"""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for step in range(1, steps + 1):
        src, src_len, tgt_in, tgt_out = make_batch(batch_size)
        logits, _ = model(src, src_len, tgt_in)
        # logits [B,T,V] 摊平成 [B*T, V] 与 tgt_out [B*T] 对齐；PAD 不计入 loss
        loss = F.cross_entropy(logits.reshape(-1, VOCAB), tgt_out.reshape(-1),
                               ignore_index=PAD)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step % log_every == 0 or step == 1:
            print(f"  step {step:4d}/{steps}  loss = {loss.item():.4f}")
    return model

@torch.no_grad()
def seq_accuracy(model, n_batches=20, batch_size=128, min_len=4, max_len=12):
    """整条序列全对才算对（exact match），统计 n_batches 个 batch 的平均准确率。"""
    model.eval()
    correct = total = 0
    for _ in range(n_batches):
        src, src_len, tgt_in, tgt_out = make_batch(batch_size, min_len, max_len)
        logits, _ = model(src, src_len, tgt_in)
        pred = logits.argmax(-1)                        # [B, T] 贪心取每步最大
        valid = tgt_out != PAD                          # 只看非 PAD 位置
        ok = ((pred == tgt_out) | ~valid).all(dim=1)    # 该样本所有有效位置都对
        correct += ok.sum().item()
        total += src.size(0)
    return correct / total

In [ ]:
# ============================================================
# Cell 8: 训练两个模型——带 attention vs 不带（baseline）（对应第 8 节）
# ============================================================
import time

torch.manual_seed(0)
print("训练【带 attention】的 seq2seq ...")
t0 = time.time()
model_attn = Seq2Seq(VOCAB, emb_dim=32, hid_dim=96, use_attention=True)
train(model_attn, steps=1500)
print(f"  用时 {time.time() - t0:.1f}s\n")

torch.manual_seed(0)
print("训练【不带 attention】的 baseline（context 固定为 encoder 末隐藏）...")
t0 = time.time()
model_plain = Seq2Seq(VOCAB, emb_dim=32, hid_dim=96, use_attention=False)
train(model_plain, steps=1500)
print(f"  用时 {time.time() - t0:.1f}s")

In [ ]:
# ============================================================
# Cell 9: 可视化 attention 对齐矩阵——逆序任务应是一条反对角线（对应第 4.3 节）
# ============================================================
import matplotlib.pyplot as plt

# 取一条定长样本，看带 attention 的模型每步在「看」源序列的哪个位置
src, src_len, tgt_in, tgt_out = make_batch(1, min_len=8, max_len=8, seed=7)
model_attn.eval()
with torch.no_grad():
    logits, attn = model_attn(src, src_len, tgt_in, return_attn=True)
pred = logits.argmax(-1)[0]                 # [T]

src_digits = (src[0] - DIGIT0).tolist()     # 还原成 0..9 便于看
L = src_len.item()
A = attn[0, :L, :L].numpy()                  # [T, S] 只取有效部分

print("源序列(数字):", src_digits[:L])
print("目标(逆序)  :", (tgt_out[0, :L]).sub(DIGIT0).tolist())
print("模型预测     :", (pred[:L]).sub(DIGIT0).tolist())

plt.figure(figsize=(5, 4))
plt.imshow(A, cmap="viridis", aspect="auto")
plt.xlabel("source position j")            # matplotlib 标签一律英文，避免豆腐块
plt.ylabel("decoder step t")
plt.title("Bahdanau Attention Alignment (reverse task)")
plt.colorbar(label="attention weight")
plt.tight_layout()
plt.show()
print("→ 亮点连成反对角线：解码第 t 步把注意力压在源序列第 (L-1-t) 个位置上，")
print("  正是逆序任务需要的对齐——attention 自己学会了『回看正确的输入位置』。")

In [ ]:
# ============================================================
# Cell 10: 瓶颈验证——准确率随序列变长，baseline 掉得比 attention 快（对应第 2.3 / 6 节）
# ============================================================
# 同样的训练量下，把测试集按长度分桶，比较两个模型的整句准确率。
# 预期：短序列两者都行；越长，无 attention 的 baseline 越吃力（信息被压进单个向量），
# 带 attention 的模型则能稳住——这就是 attention 当年被发明出来要解决的「瓶颈」。
buckets = [(4, 5), (6, 7), (8, 9), (10, 12)]
acc_attn, acc_plain = [], []
for lo, hi in buckets:
    acc_attn.append(seq_accuracy(model_attn, min_len=lo, max_len=hi))
    acc_plain.append(seq_accuracy(model_plain, min_len=lo, max_len=hi))

labels = [f"{lo}-{hi}" for lo, hi in buckets]
for lab, a, p in zip(labels, acc_attn, acc_plain):
    print(f"  长度 {lab:5s}  attention={a:.2%}   baseline={p:.2%}")

x = range(len(buckets))
plt.figure(figsize=(6, 4))
plt.plot(list(x), acc_attn, "o-", label="with attention")
plt.plot(list(x), acc_plain, "s--", label="no attention (baseline)")
plt.xticks(list(x), labels)
plt.xlabel("source length bucket")
plt.ylabel("exact-match accuracy")
plt.title("Why Attention: accuracy vs sequence length")
plt.ylim(0, 1.05)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("→ 序列越长，固定 context 的 baseline 越扛不住（瓶颈），attention 基本稳住。")